## Part A: Post Sentiment Classification ##

In [2]:
import pandas as pd
import numpy as np

train_data = pd.read_json("https://raw.githubusercontent.com/rpsoft/tad_course/main/reddit_sentiment_train.json")

validation_data = pd.read_json("https://raw.githubusercontent.com/rpsoft/tad_course/main/reddit_sentiment_validation.json")

test_data = pd.read_json("https://raw.githubusercontent.com/rpsoft/tad_course/main/reddit_sentiment_test.json")

In [3]:
# import warnings
# warnings.filterwarnings("ignore")

In [4]:
### The one_hot_vectorizer from the labs
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner'])
nlp.remove_pipe('tagger')
nlp.remove_pipe('parser')

#@Tokenize
def spacy_tokenize(string):
    tokens = list()
    doc = nlp(string)
    for token in doc:
        tokens.append(token)
    return tokens

#@Normalize
def normalize(tokens):
    normalized_tokens = list()
    for token in tokens:
        normalized = token.text.lower().strip()
        if ((token.is_alpha or token.is_digit)):
            normalized_tokens.append(normalized)
    return normalized_tokens

#@Tokenize and normalize
def tokenize_normalize(string):
    return normalize(spacy_tokenize(string))

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

one_hot_vectorizer = CountVectorizer(tokenizer = tokenize_normalize, binary=True, max_features=20000)
tfidf_vectorizer = TfidfVectorizer()

print(train_data.columns)

Index(['subreddit', 'title', 'id', 'url', 'author', 'body', 'majority_type',
       'is_first_post', 'post_depth', 'in_reply_to', 'sentiment.polarity',
       'sentiment.subjectivity'],
      dtype='object')


## Setup

In [5]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import fbeta_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import plot_roc_curve

In [6]:
import warnings
# warnings.filterwarnings("ignore")# Function to print classification report (modified from lab 4)
def evaluation_summary(description, predictions, true_labels):
    print("Evaluation for: " + description)
    precision = precision_score(predictions, true_labels, average="macro")
    recall = recall_score(predictions, true_labels, average="macro")
    accuracy = accuracy_score(predictions, true_labels)
    f1 = fbeta_score(predictions, true_labels, 1, average="macro")
    print("Classifier '%s' has Acc=%0.3f P=%0.3f R=%0.3f F1=%0.3f" % (description,accuracy,precision,recall,f1))
    print(classification_report(predictions, true_labels, digits=3))
    print('\nConfusion matrix:\n',confusion_matrix(true_labels, predictions))

In [7]:
class ItemSelector(BaseEstimator, TransformerMixin):
    """For data grouped by feature, select subset of data at a provided key.    """

    def __init__(self, key):
        self.key = key

    def fit(self, x, y=None):
        return self

    def transform(self, data_dict):
        return data_dict[self.key]

In [8]:
# Randomise data
train_data.sample(frac=1)
validation_data.sample(frac=1)
test_data.sample(frac=1)

# Create input features
train_features = one_hot_vectorizer.fit_transform(train_data['body'])
validation_features = one_hot_vectorizer.transform(validation_data['body'])
test_features = one_hot_vectorizer.transform(test_data['body'])

train_features_tfidf = tfidf_vectorizer.fit_transform(train_data['body'])
validation_features_tfidf = tfidf_vectorizer.transform(validation_data['body'])
test_features_tfidf = tfidf_vectorizer.transform(test_data['body'])

# Create label variables (for convenience)
train_labels = train_data['sentiment.polarity']
validation_labels = validation_data['sentiment.polarity']
test_labels = test_data['sentiment.polarity']

###  Data

In [9]:
print("Train Length: %d, Validation Length: %d, Test Length: %d\n" % (len(train_data), len(validation_data), len(test_data)))

print("Train Data")
print("--: %0.0f%%, -: %0.0f%%, N: %0.0f%%, +: %0.0f%%, ++: %0.0f%%\n" % (100*sum(train_data['sentiment.polarity'] == 'very negative')/len(train_data), 
                                              100*sum(train_data['sentiment.polarity'] == 'negative')/len(train_data),
                                              100*sum(train_data['sentiment.polarity'] == 'neutral')/len(train_data),
                                              100*sum(train_data['sentiment.polarity'] == 'positive')/len(train_data),
                                              100*sum(train_data['sentiment.polarity'] == 'very positive')/len(train_data)))
print("Validation Data")
print("--: %0.0f%%, -: %0.0f%%, N: %0.0f%%, +: %0.0f%%, ++: %0.0f%%\n" % (100*sum(validation_data['sentiment.polarity'] == 'very negative')/len(validation_data), 
                                              100*sum(validation_data['sentiment.polarity'] == 'negative')/len(validation_data),
                                              100*sum(validation_data['sentiment.polarity'] == 'neutral')/len(validation_data),
                                              100*sum(validation_data['sentiment.polarity'] == 'positive')/len(validation_data),
                                              100*sum(validation_data['sentiment.polarity'] == 'very positive')/len(validation_data)))
print("Test Data")
print("--: %0.0f%%, -: %0.0f%%, N: %0.0f%%, +: %0.0f%%, ++: %0.0f%%\n" % (100*sum(test_data['sentiment.polarity'] == 'very negative')/len(test_data), 
                                              100*sum(test_data['sentiment.polarity'] == 'negative')/len(test_data),
                                              100*sum(test_data['sentiment.polarity'] == 'neutral')/len(test_data),
                                              100*sum(test_data['sentiment.polarity'] == 'positive')/len(test_data),
                                              100*sum(test_data['sentiment.polarity'] == 'very positive')/len(test_data)))

Train Length: 12138, Validation Length: 3109, Test Length: 4016

Train Data
--: 1%, -: 7%, N: 63%, +: 27%, ++: 2%

Validation Data
--: 0%, -: 7%, N: 63%, +: 27%, ++: 2%

Test Data
--: 1%, -: 7%, N: 63%, +: 27%, ++: 2%



### 1a) Dummy Classifier - Most Frequant

In [10]:
# Create baseline classifier and fit data
dummy_mf = DummyClassifier(strategy='most_frequent')
dummy_mf.fit(train_features, train_labels)

DummyClassifier(strategy='most_frequent')

In [85]:
# Print evaluation
evaluation_summary("Dummy MF", dummy_mf.predict(test_features), test_labels)
evaluation_summary("Dummy MF", dummy_mf.predict(train_features), train_labels)
evaluation_summary("Dummy MF", dummy_mf.predict(validation_features), validation_labels)

Evaluation for: Dummy MF
Classifier 'Dummy MF' has Acc=0.626 P=0.200 R=0.125 F1=0.154
               precision    recall  f1-score   support

     negative      0.000     0.000     0.000         0
      neutral      1.000     0.626     0.770      4016
     positive      0.000     0.000     0.000         0
very negative      0.000     0.000     0.000         0
very positive      0.000     0.000     0.000         0

     accuracy                          0.626      4016
    macro avg      0.200     0.125     0.154      4016
 weighted avg      1.000     0.626     0.770      4016


Confusion matrix:
 [[   0  282    0    0    0]
 [   0 2514    0    0    0]
 [   0 1102    0    0    0]
 [   0   32    0    0    0]
 [   0   86    0    0    0]]


### 1b) Dummy Classifier - Stratified

In [173]:
# Create baseline classifier and fit data
dummy_strat = DummyClassifier(strategy='stratified')
dummy_strat.fit(train_features_b, train_labels_b)

DummyClassifier(strategy='stratified')

In [174]:
# Print evaluation
evaluation_summary("Dummy S", dummy_strat.predict(test_features), test_labels)
evaluation_summary("Dummy S", dummy_strat.predict(train_features), train_labels)
evaluation_summary("Dummy S", dummy_strat.predict(validation_features), validation_labels)

Evaluation for: Dummy S
Classifier 'Dummy S' has Acc=0.260 P=0.195 R=0.197 F1=0.158
               precision    recall  f1-score   support

     negative      0.238     0.063     0.100      1064
      neutral      0.267     0.623     0.374      1079
     positive      0.262     0.268     0.265      1078
very negative      0.031     0.005     0.009       200
very positive      0.174     0.025     0.044       595

     accuracy                          0.260      4016
    macro avg      0.195     0.197     0.158      4016
 weighted avg      0.233     0.260     0.205      4016


Confusion matrix:
 [[ 67  81  68  20  46]
 [664 672 687 114 377]
 [302 299 289  60 152]
 [ 12   3  11   1   5]
 [ 19  24  23   5  15]]


### 1c) Logistic Regression - One-Hot Vectorization

In [175]:
# Create model
lr_onehot = LogisticRegression(max_iter=1000)
lr_onehot_model = lr_onehot.fit(train_features, train_labels)

In [176]:
# Print evaluation of model
evaluation_summary("LR One-Hot", lr_onehot_model.predict(test_features), test_labels)
evaluation_summary("LR One-Hot", lr_onehot_model.predict(train_features), train_labels)
evaluation_summary("LR One-Hot", lr_onehot_model.predict(validation_features), validation_labels)

Evaluation for: LR One-Hot
Classifier 'LR One-Hot' has Acc=0.748 P=0.439 R=0.632 F1=0.486
               precision    recall  f1-score   support

     negative      0.248     0.486     0.329       144
      neutral      0.876     0.779     0.825      2828
     positive      0.635     0.710     0.670       986
very negative      0.156     0.714     0.256         7
very positive      0.279     0.471     0.350        51

     accuracy                          0.748      4016
    macro avg      0.439     0.632     0.486      4016
 weighted avg      0.786     0.748     0.762      4016


Confusion matrix:
 [[  70  202    9    1    0]
 [  66 2203  232    1   12]
 [   3  384  700    0   15]
 [   5   22    0    5    0]
 [   0   17   45    0   24]]


### 1d) Logistic Regression - TF-IDF Vectorization

In [16]:
# Create model
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf_model = lr_tfidf.fit(train_features_tfidf, train_labels)

In [89]:
# Print evaluation of model
evaluation_summary("LR TFIDF", lr_tfidf_model.predict(test_features_tfidf), test_labels)
evaluation_summary("LR TFIDF", lr_tfidf_model.predict(train_features_tfidf), train_labels)
evaluation_summary("LR TFIDF", lr_tfidf_model.predict(validation_features_tfidf), validation_labels)

Evaluation for: LR TFIDF
Classifier 'LR TFIDF' has Acc=0.739 P=0.325 R=0.578 F1=0.349
               precision    recall  f1-score   support

     negative      0.092     0.619     0.160        42
      neutral      0.944     0.736     0.827      3227
     positive      0.508     0.759     0.609       738
very negative      0.000     0.000     0.000         0
very positive      0.081     0.778     0.147         9

     accuracy                          0.739      4016
    macro avg      0.325     0.578     0.349      4016
 weighted avg      0.853     0.739     0.778      4016


Confusion matrix:
 [[  26  249    7    0    0]
 [  12 2374  127    0    1]
 [   0  541  560    0    1]
 [   4   28    0    0    0]
 [   0   35   44    0    7]]


### 1e) SVC - One-Hot Vectorization

In [179]:
# Create model
svc = SVC()
svc_model = svc.fit(train_features, train_labels)

In [180]:
# Print evaluation of model
evaluation_summary("SVC One-Hot", svc_model.predict(test_features), test_labels)
evaluation_summary("SVC One-Hot", svc_model.predict(train_features), train_labels)
evaluation_summary("SVC One-Hot", svc_model.predict(validation_features), validation_labels)

Evaluation for: SVC One-Hot
Classifier 'SVC One-Hot' has Acc=0.730 P=0.288 R=0.459 F1=0.287
               precision    recall  f1-score   support

     negative      0.014     0.800     0.028         5
      neutral      0.959     0.721     0.823      3342
     positive      0.469     0.773     0.584       669
very negative      0.000     0.000     0.000         0
very positive      0.000     0.000     0.000         0

     accuracy                          0.730      4016
    macro avg      0.288     0.459     0.287      4016
 weighted avg      0.876     0.730     0.783      4016


Confusion matrix:
 [[   4  270    8    0    0]
 [   0 2411  103    0    0]
 [   0  585  517    0    0]
 [   1   31    0    0    0]
 [   0   45   41    0    0]]


### 1f) Neural Network - Bag-of-Words

In [20]:
import warnings
# warnings.filterwarnings("ignore")# Pre-Process
bag_of_words_vectorizer = CountVectorizer(tokenizer=tokenize_normalize, binary=False)

train_features_bow = bag_of_words_vectorizer.fit_transform(train_data['body'])
validation_features_bow = bag_of_words_vectorizer.transform(validation_data['body'])
test_features_bow = bag_of_words_vectorizer.transform(test_data['body'])

# Create and train model
neural_net = MLPClassifier(verbose=True)
neural_net_model = neural_net.fit(train_features_bow, train_labels)

Iteration 1, loss = 1.10520474
Iteration 2, loss = 0.76208033
Iteration 3, loss = 0.53500436
Iteration 4, loss = 0.37981083
Iteration 5, loss = 0.27242108
Iteration 6, loss = 0.19682039
Iteration 7, loss = 0.14365296
Iteration 8, loss = 0.10826088
Iteration 9, loss = 0.08369515
Iteration 10, loss = 0.06601615
Iteration 11, loss = 0.05301784
Iteration 12, loss = 0.04358822
Iteration 13, loss = 0.03652926
Iteration 14, loss = 0.03107567
Iteration 15, loss = 0.02689575
Iteration 16, loss = 0.02369308
Iteration 17, loss = 0.02103765
Iteration 18, loss = 0.01891076
Iteration 19, loss = 0.01715893
Iteration 20, loss = 0.01571960
Iteration 21, loss = 0.01452997
Iteration 22, loss = 0.01355005
Iteration 23, loss = 0.01276840
Iteration 24, loss = 0.01197124
Iteration 25, loss = 0.01136529
Iteration 26, loss = 0.01076882
Iteration 27, loss = 0.01028877
Iteration 28, loss = 0.00992078
Iteration 29, loss = 0.00953853
Iteration 30, loss = 0.00919024
Iteration 31, loss = 0.00887596
Iteration 32, los

In [80]:
# Print evaluation of model
evaluation_summary("Neural Network BOW", neural_net_model.predict(test_features_bow), test_labels)
evaluation_summary("Neural Network BOW", neural_net_model.predict(train_features_bow), train_labels)
evaluation_summary("Neural Network BOW", neural_net_model.predict(validation_features_bow), validation_labels)

Evaluation for: Neural Network BOW
Classifier 'Neural Network BOW' has Acc=0.747 P=0.521 R=0.588 F1=0.549
               precision    recall  f1-score   support

     negative      0.337     0.495     0.401       192
      neutral      0.850     0.793     0.821      2694
     positive      0.657     0.700     0.678      1035
very negative      0.375     0.480     0.421        25
very positive      0.384     0.471     0.423        70

     accuracy                          0.747      4016
    macro avg      0.521     0.588     0.549      4016
 weighted avg      0.765     0.747     0.754      4016


Confusion matrix:
 [[  95  176    8    3    0]
 [  83 2137  272    9   13]
 [   9  344  724    1   24]
 [   5   15    0   12    0]
 [   0   22   31    0   33]]
Evaluation for: Neural Network BOW
Classifier 'Neural Network BOW' has Acc=0.999 P=0.994 R=0.999 F1=0.996
               precision    recall  f1-score   support

     negative      1.000     1.000     1.000       878
      neutral     

### 2) Logistic Regression - TF-IDF Vectorization Paramater Tuning

In [10]:
lr_tfidf_pipeline = Pipeline([
              ('selector', ItemSelector(key='body')),
              ('tfidf', TfidfVectorizer()),
              ('logreg', LogisticRegression(max_iter=1000))
              ])
lr_tfidf_pipeline.fit(train_data, train_labels)
evaluation_summary("LR TFIDF", lr_tfidf_pipeline.predict(validation_data), validation_labels)   

Evaluation for: LR TFIDF
Classifier 'LR TFIDF' has Acc=0.731 P=0.320 R=0.564 F1=0.344
               precision    recall  f1-score   support

     negative      0.084     0.581     0.146        31
      neutral      0.938     0.732     0.823      2512
     positive      0.482     0.731     0.581       557
very negative      0.000     0.000     0.000         0
very positive      0.096     0.778     0.171         9

     accuracy                          0.731      3109
    macro avg      0.320     0.564     0.344      3109
 weighted avg      0.846     0.731     0.771      3109


Confusion matrix:
 [[  18  193    4    0    0]
 [  10 1840  110    0    1]
 [   1  436  407    0    1]
 [   2   13    0    0    0]
 [   0   30   36    0    7]]


In [12]:
# round 1

params = {
    'logreg__C': (0.001, 0, 10000, 100000), 
    'logreg__solver': ('newton-cg', 'sag', 'saga', 'lbfgs'),
    'tfidf__sublinear_tf': (True, False),
    'tfidf__max_features': (5000, 10000, 15000, 20000),
}

grid_search = GridSearchCV(lr_tfidf_pipeline, param_grid=params, n_jobs=1, verbose=1, scoring='f1_macro', cv=5)
grid_search.fit(train_data, train_labels)

print("Best score: %0.3f" % grid_search.best_score_)
print("Best parameters set:")
best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(params.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best score: 0.490
Best parameters set:
	logreg__C: 10000
	logreg__solver: 'saga'
	tfidf__max_features: 10000
	tfidf__sublinear_tf: True


In [13]:
# round 2

params = {
    'logreg__C': (0, 500, 1000, 10000), 
    'logreg__solver': (['lbfgs']),
    'tfidf__sublinear_tf': ([True]),
    'tfidf__max_features': (100, 2000, 5000, 10000),
}

grid_search = GridSearchCV(lr_tfidf_pipeline, param_grid=params, n_jobs=1, verbose=1, scoring='f1_macro', cv=5)
grid_search.fit(train_data, train_labels)

print("Best score: %0.3f" % grid_search.best_score_)
print("Best parameters set:")
best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(params.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best score: 0.498
Best parameters set:
	logreg__C: 500
	logreg__solver: 'lbfgs'
	tfidf__max_features: 10000
	tfidf__sublinear_tf: True


In [64]:
# round 3

params = {
    'logreg__C': (0, 250, 500, 600), 
    'logreg__solver': (['lbfgs']),
    'tfidf__sublinear_tf': ([True]),
    'tfidf__max_features': (3000, 4000, 5000, 6000),
}

grid_search = GridSearchCV(lr_tfidf_pipeline, param_grid=params, n_jobs=1, verbose=1, scoring='f1_macro', cv=5)
grid_search.fit(train_data, train_labels)

print("Best score: %0.3f" % grid_search.best_score_)
print("Best parameters set:")
best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(params.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best score: 0.507
Best parameters set:
	logreg__C: 500
	logreg__solver: 'lbfgs'
	tfidf__max_features: 6000
	tfidf__sublinear_tf: True


In [65]:
evaluation_summary("LR TFIDF", grid_search.predict(test_data), test_labels)   

Evaluation for: LR TFIDF
Classifier 'LR TFIDF' has Acc=0.744 P=0.498 R=0.563 F1=0.524
               precision    recall  f1-score   support

     negative      0.337     0.473     0.393       201
      neutral      0.831     0.803     0.817      2600
     positive      0.697     0.680     0.688      1129
very negative      0.344     0.478     0.400        23
very positive      0.279     0.381     0.322        63

     accuracy                          0.744      4016
    macro avg      0.498     0.563     0.524      4016
 weighted avg      0.757     0.744     0.749      4016


Confusion matrix:
 [[  95  172    9    4    2]
 [  94 2089  305    8   18]
 [   5  310  768    0   19]
 [   7   12    2   11    0]
 [   0   17   45    0   24]]


In [202]:
error_analysis = validation_data.copy(deep=True)
error_analysis['predicted'] = grid_search.predict(validation_data)


### Used this code for error analysis
# with open('error analysis.txt', 'w') as f:
#     f.write(error_analysis.to_string(columns=['body', 'sentiment.polarity', 'predicted']))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 body sentiment.polarity      predicted
0                                                                                                                                                                                                                                                                                               

### 3) Logistic Regression - TF-IDF Vectorization with Added Features

In [115]:
lr_tfidf_union_pipeline = Pipeline([
        ('union', FeatureUnion(
          transformer_list=[
            ('body-tfidf', Pipeline([
              ('selector', ItemSelector(key='body')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True, max_features=6000)), 
              ])),
            ('body-oh', Pipeline([
              ('selector', ItemSelector(key='body')),
              ('one-hot', CountVectorizer(tokenizer=tokenize_normalize, binary=True, max_features=5000)), 
              ])),
            ('majority_type', Pipeline([
              ('selector', ItemSelector(key='majority_type')),
              ('tfidf', CountVectorizer(tokenizer=tokenize_normalize, binary=True, max_features=5000)), 
              ])),
        ])
        )
    ])

pipeline_train_features = lr_tfidf_union_pipeline.fit_transform(train_data)
pipeline_validation_features = lr_tfidf_union_pipeline.transform(validation_data)
pipeline_test_features = lr_tfidf_union_pipeline.transform(test_data)

In [116]:
# Create model
lr_tfidf_af = LogisticRegression(max_iter=1000, C=500)
lr_tfidf_model_af = lr_tfidf_af.fit(pipeline_train_features, train_labels)

In [117]:
# Print evaluation of model
evaluation_summary("LR TFIDF w/ Added Features", lr_tfidf_model_af.predict(pipeline_test_features), test_labels)

Evaluation for: LR TFIDF w/ Added Features
Classifier 'LR TFIDF w/ Added Features' has Acc=0.740 P=0.539 R=0.543 F1=0.541
               precision    recall  f1-score   support

     negative      0.418     0.449     0.433       263
      neutral      0.813     0.813     0.813      2514
     positive      0.696     0.684     0.690      1121
very negative      0.406     0.406     0.406        32
very positive      0.360     0.360     0.360        86

     accuracy                          0.740      4016
    macro avg      0.539     0.543     0.541      4016
 weighted avg      0.742     0.740     0.741      4016


Confusion matrix:
 [[ 118  149    8    6    1]
 [ 128 2044  306   12   24]
 [   9  295  767    1   30]
 [   7   12    0   13    0]
 [   1   14   40    0   31]]


## Part B: Thread Subreddit Prediction ##

In [119]:
import pandas as pd

subreddit_train_data = pd.read_json("https://raw.githubusercontent.com/rpsoft/tad_course/main/reddit_discourse_train.json")
subreddit_test_data = pd.read_json("https://raw.githubusercontent.com/rpsoft/tad_course/main/reddit_discourse_test.json")

# Top 20 subreddits are filtered here for you.
top_subreddits = subreddit_train_data.subreddit.value_counts().head(20)

subreddit_train_data = subreddit_train_data[subreddit_train_data['subreddit'].isin(top_subreddits.keys())]
subreddit_test_data = subreddit_test_data[subreddit_test_data['subreddit'].isin(top_subreddits.keys())]

subreddit_train_labels = subreddit_train_data['subreddit']
subreddit_test_labels = subreddit_test_data['subreddit']

#### Training / Validation data

For part B the training is to be performed in a utilising cross-validation. See Lab 4 for an example, in particular the use of GridSearchCV.

In [25]:
subreddit_feature_pipeline = Pipeline([
        ('union', FeatureUnion(
          transformer_list=[
            ('title', Pipeline([
              ('selector', ItemSelector(key='title')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
            ('body', Pipeline([
              ('selector', ItemSelector(key='body')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
            ('author', Pipeline([
              ('selector', ItemSelector(key='author')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
        ])
        )
    ])

subreddit_pipeline_train_features = subreddit_feature_pipeline.fit_transform(subreddit_train_data)
subreddit_pipeline_test_features = subreddit_feature_pipeline.transform(subreddit_test_data)

In [143]:
added_feature = Pipeline([
        ('union', FeatureUnion(
          transformer_list=[
            ('title', Pipeline([
              ('selector', ItemSelector(key='title')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
            ('body', Pipeline([
              ('selector', ItemSelector(key='body')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
            ('author', Pipeline([
              ('selector', ItemSelector(key='author')),
              ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
              ])),
            ('body-bigram', Pipeline([
              ('selector', ItemSelector(key='body')),
              ('one-hot', CountVectorizer(tokenizer=tokenize_normalize, binary=True, max_features=5000, ngram_range=(2,2))), 
              ])),
            ('sentiment-polarity', Pipeline([
              ('selector', ItemSelector(key='sentiment.polarity')),
              ('one-hot', CountVectorizer(tokenizer=tokenize_normalize, binary=True, ngram_range=(1,2))),
              ])),
        ])
        )
])

subreddit_pipeline2_train_features = added_feature.fit_transform(subreddit_train_data)
subreddit_pipeline2_test_features = added_feature.transform(subreddit_test_data)

### Logistic Regression

In [26]:
# Create model
subreddit_lr = LogisticRegression(max_iter=1000, C=10)  # L2 regularisation is default
subreddit_lr_model = subreddit_lr.fit(subreddit_pipeline_train_features, subreddit_train_labels)

In [27]:
# Print evaluation of model
evaluation_summary("Subreddit LR", subreddit_lr_model.predict(subreddit_pipeline_test_features), subreddit_test_labels)

Evaluation for: Subreddit LR
Classifier 'Subreddit LR' has Acc=0.592 P=0.444 R=0.622 F1=0.472
                      precision    recall  f1-score   support

           askreddit      0.910     0.594     0.719      1557
             atheism      0.164     0.610     0.259        41
            buildapc      0.829     0.773     0.800       370
electronic_cigarette      0.545     0.711     0.617        76
   explainlikeimfive      0.992     0.983     0.987       119
              gaming      0.145     0.442     0.218        52
                guns      0.157     1.000     0.271        19
              hockey      0.258     0.769     0.386        52
     leagueoflegends      0.672     0.448     0.537       616
         legaladvice      0.667     0.833     0.741        12
              loseit      0.233     0.636     0.341        11
              movies      0.224     0.314     0.262        35
        pcmasterrace      0.147     0.209     0.172       134
     personalfinance      0.731     0

In [137]:
subreddit_lr2 = LogisticRegression(max_iter=1000)

parameters_lr = {
    'C': (0.001, 500, 5000, 50000), 
    'solver': ('newton-cg', 'sag', 'saga', 'lbfgs'),
}

lr_cv = GridSearchCV(subreddit_lr2, parameters_lr, cv=3)

GridSearchCV(cv=3, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': (0.001, 0, 10000, 100000),
                         'solver': ('newton-cg', 'sag', 'saga', 'lbfgs')})

In [144]:
lr_cv.fit(subreddit_pipeline2_train_features, subreddit_train_labels)

GridSearchCV(cv=3, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': (0.001, 0, 10000, 100000),
                         'solver': ('newton-cg', 'sag', 'saga', 'lbfgs')})

In [145]:
evaluation_summary("Subreddit LR", lr_cv.predict(subreddit_pipeline2_test_features), subreddit_test_labels)

Evaluation for: Subreddit LR
Classifier 'Subreddit LR' has Acc=0.586 P=0.438 R=0.542 F1=0.458
                      precision    recall  f1-score   support

           askreddit      0.905     0.628     0.741      1464
             atheism      0.178     0.519     0.265        52
            buildapc      0.794     0.714     0.752       384
electronic_cigarette      0.525     0.565     0.545        92
   explainlikeimfive      0.966     0.958     0.962       119
              gaming      0.170     0.338     0.226        80
                guns      0.149     0.818     0.252        22
              hockey      0.452     0.680     0.543       103
     leagueoflegends      0.650     0.479     0.552       557
         legaladvice      0.600     0.600     0.600        15
              loseit      0.233     0.412     0.298        17
              movies      0.408     0.392     0.400        51
        pcmasterrace      0.157     0.214     0.181       140
     personalfinance      0.538     0

### SVC

In [28]:
# Create model
subreddit_svc = SVC()
subreddit_svc_model = svc.fit(subreddit_pipeline_train_features, subreddit_train_labels)

In [29]:
# Print evaluation of model
evaluation_summary("Subreddit SVC", subreddit_svc_model.predict(subreddit_pipeline_test_features), subreddit_test_labels)

Evaluation for: Subreddit SVC
Classifier 'Subreddit SVC' has Acc=0.553 P=0.341 R=0.651 F1=0.358
                      precision    recall  f1-score   support

           askreddit      0.930     0.553     0.694      1708
             atheism      0.046     0.636     0.086        11
            buildapc      0.817     0.775     0.795       364
electronic_cigarette      0.394     0.867     0.542        45
   explainlikeimfive      0.992     1.000     0.996       117
              gaming      0.101     0.667     0.175        24
                guns      0.000     0.000     0.000         0
              hockey      0.013     1.000     0.025         2
     leagueoflegends      0.706     0.364     0.481       796
         legaladvice      0.067     1.000     0.125         1
              loseit      0.033     1.000     0.065         1
              movies      0.265     0.260     0.263        50
        pcmasterrace      0.126     0.226     0.162       106
     personalfinance      0.404    

In [151]:
subreddit_svc2 = SVC()

parameters_svc = {
    'C': (0.001, 0, 10000, 100000), 
    'kernel': ('linear', 'rbf', 'sigmoid'),
}

svc_cv = GridSearchCV(subreddit_svc2, parameters_svc, cv=3)

In [152]:
svc_cv.fit(subreddit_pipeline2_train_features, subreddit_train_labels)

GridSearchCV(cv=3, estimator=SVC(),
             param_grid={'C': (0.001, 0, 10000, 100000),
                         'kernel': ('linear', 'rbf', 'sigmoid')})

In [153]:
evaluation_summary("Subreddit SVC", svc_cv.predict(subreddit_pipeline2_test_features), subreddit_test_labels)

Evaluation for: Subreddit SVC
Classifier 'Subreddit SVC' has Acc=0.538 P=0.354 R=0.561 F1=0.379
                      precision    recall  f1-score   support

           askreddit      0.911     0.571     0.702      1621
             atheism      0.086     0.684     0.152        19
            buildapc      0.762     0.731     0.746       360
electronic_cigarette      0.364     0.643     0.465        56
   explainlikeimfive      0.949     0.991     0.970       113
              gaming      0.145     0.390     0.211        59
                guns      0.008     0.500     0.016         2
              hockey      0.039     0.857     0.074         7
     leagueoflegends      0.693     0.359     0.473       794
         legaladvice      0.400     0.462     0.429        13
              loseit      0.200     0.429     0.273        14
              movies      0.367     0.321     0.343        56
        pcmasterrace      0.147     0.219     0.176       128
     personalfinance      0.337    

### Neural Network

In [30]:
# create model
subreddit_neural_net = MLPClassifier(verbose=True)
subreddit_neural_net_model = subreddit_neural_net.fit(subreddit_pipeline_train_features, subreddit_train_labels)

Iteration 1, loss = 2.35576140
Iteration 2, loss = 0.80902975
Iteration 3, loss = 0.18550606
Iteration 4, loss = 0.06847392
Iteration 5, loss = 0.03655899
Iteration 6, loss = 0.02341474
Iteration 7, loss = 0.01668450
Iteration 8, loss = 0.01271693
Iteration 9, loss = 0.01017972
Iteration 10, loss = 0.00844657
Iteration 11, loss = 0.00720456
Iteration 12, loss = 0.00628182
Iteration 13, loss = 0.00557857
Iteration 14, loss = 0.00502300
Iteration 15, loss = 0.00458163
Iteration 16, loss = 0.00422088
Iteration 17, loss = 0.00392593
Iteration 18, loss = 0.00367816
Iteration 19, loss = 0.00347074
Iteration 20, loss = 0.00329476
Iteration 21, loss = 0.00314412
Iteration 22, loss = 0.00301416
Iteration 23, loss = 0.00290050
Iteration 24, loss = 0.00280091
Iteration 25, loss = 0.00271357
Iteration 26, loss = 0.00263528
Iteration 27, loss = 0.00256590
Iteration 28, loss = 0.00250348
Iteration 29, loss = 0.00244697
Iteration 30, loss = 0.00239472
Iteration 31, loss = 0.00234754
Iteration 32, los

In [31]:
# Print evaluation of model
evaluation_summary("Neural Network", subreddit_neural_net_model.predict(subreddit_pipeline_test_features), subreddit_test_labels)

Evaluation for: Neural Network
Classifier 'Neural Network' has Acc=0.614 P=0.484 R=0.601 F1=0.500
                      precision    recall  f1-score   support

           askreddit      0.904     0.587     0.711      1565
             atheism      0.191     0.518     0.279        56
            buildapc      0.872     0.729     0.794       413
electronic_cigarette      0.636     0.708     0.670        89
   explainlikeimfive      0.695     0.752     0.722       109
              gaming      0.176     0.609     0.273        46
                guns      0.207     1.000     0.342        25
              hockey      0.684     0.898     0.777       118
     leagueoflegends      0.681     0.591     0.633       474
         legaladvice      0.867     0.481     0.619        27
              loseit      0.433     0.867     0.578        15
              movies      0.510     0.321     0.394        78
        pcmasterrace      0.147     0.354     0.207        79
     personalfinance      0.712  

In [183]:
subreddit_neural_net2 = MLPClassifier()

parameters_neural_net = {
    'hidden_layer_sizes': ((50,), (100,), (120,)), 
    'solver': ('lbfgs', 'sgd', 'adam'),
}

# reduced cv from 3 to 2 for neural network so the
# time taken to complete isn't ridiculous
neural_net_cv = GridSearchCV(subreddit_neural_net2, parameters_neural_net, cv=2)
neural_net_cv.fit(subreddit_pipeline2_train_features, subreddit_train_labels)

GridSearchCV(cv=2, estimator=MLPClassifier(),
             param_grid={'hidden_layer_sizes': ((50,), (100,), (120,)),
                         'solver': ('lbfgs', 'sgd', 'adam')})

In [184]:
evaluation_summary("Subreddit Neural Net", neural_net_cv.predict(subreddit_pipeline2_test_features), subreddit_test_labels)

Evaluation for: Subreddit Neural Net
Classifier 'Subreddit Neural Net' has Acc=0.598 P=0.458 R=0.603 F1=0.485
                      precision    recall  f1-score   support

           askreddit      0.910     0.584     0.712      1583
             atheism      0.224     0.667     0.335        51
            buildapc      0.841     0.718     0.774       404
electronic_cigarette      0.596     0.670     0.631        88
   explainlikeimfive      0.610     0.706     0.655       102
              gaming      0.157     0.424     0.229        59
                guns      0.182     0.846     0.299        26
              hockey      0.523     0.931     0.669        87
     leagueoflegends      0.667     0.536     0.594       511
         legaladvice      0.867     0.565     0.684        23
              loseit      0.400     0.500     0.444        24
              movies      0.306     0.234     0.265        64
        pcmasterrace      0.152     0.269     0.194       108
     personalfinance 

### Decision Tree

In [146]:
# train classifier
subreddit_decision_tree = DecisionTreeClassifier()
subreddit_decision_tree_model = subreddit_decision_tree.fit(subreddit_pipeline_train_features, subreddit_train_labels)

In [147]:
# Print evaluation of model
evaluation_summary("Subreddit Decision Tree", subreddit_decision_tree_model.predict(subreddit_pipeline_test_features), subreddit_test_labels)

Evaluation for: Subreddit Decision Tree
Classifier 'Subreddit Decision Tree' has Acc=0.412 P=0.239 R=0.274 F1=0.243
                      precision    recall  f1-score   support

           askreddit      0.713     0.565     0.630      1282
             atheism      0.026     0.111     0.043        36
            buildapc      0.814     0.721     0.765       390
electronic_cigarette      0.162     0.195     0.177        82
   explainlikeimfive      1.000     0.992     0.996       119
              gaming      0.000     0.000     0.000        66
                guns      0.182     0.229     0.203        96
              hockey      0.245     0.826     0.378        46
     leagueoflegends      0.345     0.213     0.263       667
         legaladvice      0.133     0.083     0.103        24
              loseit      0.033     0.048     0.039        21
              movies      0.000     0.000     0.000        27
        pcmasterrace      0.099     0.094     0.096       203
     personalfi

In [149]:
subreddit_decision_tree2 = DecisionTreeClassifier()

parameters_decision_tree = {
    'criterion': ('gini', 'entropy'), 
    'splitter': ('best', 'random'),
}

decision_tree_cv = GridSearchCV(subreddit_decision_tree2, parameters_decision_tree, cv=3)
decision_tree_cv.fit(subreddit_pipeline2_train_features, subreddit_train_labels)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ('gini', 'entropy'),
                         'splitter': ('best', 'random')})

In [150]:
evaluation_summary("Subreddit Decision Tree", decision_tree_cv.predict(subreddit_pipeline2_test_features), subreddit_test_labels)

Evaluation for: Subreddit Decision Tree
Classifier 'Subreddit Decision Tree' has Acc=0.452 P=0.309 R=0.345 F1=0.309
                      precision    recall  f1-score   support

           askreddit      0.778     0.637     0.700      1240
             atheism      0.092     0.264     0.137        53
            buildapc      0.765     0.774     0.770       341
electronic_cigarette      0.131     0.090     0.107       144
   explainlikeimfive      1.000     1.000     1.000       118
              gaming      0.025     0.056     0.035        71
                guns      0.000     0.000     0.000        17
              hockey      0.245     0.543     0.338        70
     leagueoflegends      0.387     0.238     0.295       668
         legaladvice      0.000     0.000     0.000        44
              loseit      0.467     0.560     0.509        25
              movies      0.592     0.322     0.417        90
        pcmasterrace      0.199     0.184     0.191       207
     personalfi